    This juptyer notebook depicts a query that I wrote to practice multipl joins. This problem was a more advanced join than I have done in the past and I wanted to make this a part of my portfolio becasue I learned some specific skills that I would not of learned otherwise. SOURCE: This problem was complete from HackerRank.com.  
    
    PROBLEM STATEMENT: Samantha interviews many candidates from different colleges using coding challenges and contests. Write a query to print the contest_id, hacker_id, name, and the sums of total_submissions, total_accepted_submissions, total_views, and total_unique_views for each contest sorted by contest_id. Exclude the contest from the result if all four sums are 0.

    TABLES: 
        1. contests (contest_id integer, hacker_id integer, name text)
        2. colleges (college_id integer, contest_id integer)
        3. challenges (challenge_id integer, college_id integer
        4. view_stats (challenge_id integer, total_views integer, total_unique_views integer)
        5. submission_stats (challenge_id integer, total_submissions, total_accepted_submissions)
    

    WHAT I LEARNED: 
        I had some diffifulty with getting the correct output after joining the tables together. In the first submission that I had the join statements looked like this: 
        

In [ ]:
INNER JOIN colleges on contests.contest_id = colleges.contest_id
INNER JOIN challenges on colleges.college_id = challenges.college_id
LEFT JOIN view_stats ON challenges.challenge_id = view_stats.challenge_id
LEFT JOIN submission_ stats ON challenges.challenge_id = submission_stats.challenge_id

    And the output from the query with these join statements showed this: 

In [8]:
import sqlite3
import pandas as pd 

con = sqlite3.connect("interview.db3")

ff = pd.read_sql_query("""SELECT contests.contest_id AS contest_id, 
        contests.hacker_id AS hacker_id, 
        contests.name as name,
        SUM(submission_stats.total_submissions) AS total_subs, 
        SUM(submission_stats.total_accepted_submissions) AS total_accepted_subs,
        SUM(view_stats.total_views) AS total_views,
        SUM(view_stats.total_unique_views) AS total_unique_views
FROM contests
INNER JOIN colleges on contests.contest_id = colleges.contest_id
INNER JOIN challenges on colleges.college_id = challenges.college_id
LEFT JOIN view_stats ON challenges.challenge_id = view_stats.challenge_id
LEFT JOIN submission_stats ON challenges.challenge_id = submission_stats.challenge_id
    
GROUP BY contests.contest_id, contests.hacker_id, contests.name
HAVING  SUM(submission_stats.total_submissions) != 0 AND
        SUM(submission_stats.total_accepted_submissions)!= 0 AND
        SUM(view_stats.total_views)!= 0 AND
        SUM(view_stats.total_unique_views) != 0
ORDER BY contests.contest_id""", con)

print(ff)


   contest_id hacker_id       name  total_subs  total_accepted_subs  \
0       10568      8802      Kelly        5363                 1826   
1       11100      8809      Robin        3701                 1216   
2       12742      9203      Ralph        3443                  866   
3       12861      9644     Gloria        2947                  964   
4       12865     10108     Victor        3364                  966   
5       13503     10803      David        1109                  314   
6       13537     11390      Joyce        3079                 1133   
7       13612     12592      Donna        3698                 1045   
8       14502     12923   Michelle        3673                 1148   
9       14867     13017  Stephanie        4609                 1351   
10      15164     13256     Gerald        4772                 1603   
11      15804     13421     Walter        3317                 1046   
12      15891     13569  Christina        4779                 1570   
13    

Now, the output looks like it should be correct but I was not getting the appropriate output. I double and triple checked my join statements for accuaracy and I started thinking about how I have never compiled aggregations at the same time I was making multiple joins to this extent. I found some helpful information on stack overflow that stated that I need to "pre-aggregate" the data before joining the data to the new table for a further aggregation. If I do not pre-aggregate, essentially  the data is being "counted" twice and the values will be higher than they should be. 

For the correct query below, I utilized subquires within my join statements to "pre-aggregate" and this led to the correct output that you see below. 

In [ ]:
SELECT contests.contest_id, 
        contests.hacker_id, 
        contests.name,
        SUM(submission_stats.total_submissions), 
        SUM(submission_stats.total_accepted_submissions),
        SUM(view_stats.total_views),
        SUM(view_stats.total_unique_views)
FROM contests

INNER JOIN colleges on contests.contest_id = colleges.contest_id
INNER JOIN challenges on colleges.college_id = challenges.college_id
LEFT JOIN (SELECT challenge_id, 
                    SUM(total_views) AS total_views, 
                    SUM(total_unique_views) AS total_unique_views 
            FROM view_stats 
            GROUP BY challenge_id) AS view_stats
    ON challenges.challenge_id = view_stats.challenge_id
LEFT JOIN (SELECT challenge_id, 
            SUM(total_submissions) AS total_submissions, 
            SUM(total_accepted_submissions) AS total_accepted_submissions 
            FROM Submission_Stats 
            GROUP BY challenge_id) AS submission_stats 
    ON challenges.challenge_id = submission_stats.challenge_id


GROUP BY contests.contest_id, contests.hacker_id, contests.name
HAVING  SUM(submission_stats.total_submissions) != 0 AND
        SUM(submission_stats.total_accepted_submissions)!= 0 AND
        SUM(view_stats.total_views)!= 0 AND
        SUM(view_stats.total_unique_views) != 0
ORDER BY contests.contest_id

In [9]:
import sqlite3
import pandas as pd 

con = sqlite3.connect("interview.db3")

df = pd.read_sql_query("""SELECT contests.contest_id AS contest_id, 
        contests.hacker_id AS hacker_id, 
        contests.name as name,
        SUM(submission_stats.total_submissions) AS total_subs, 
        SUM(submission_stats.total_accepted_submissions) AS total_accepted_subs,
        SUM(view_stats.total_views) AS total_views,
        SUM(view_stats.total_unique_views) AS total_unique_views
FROM contests
INNER JOIN colleges on contests.contest_id = colleges.contest_id
INNER JOIN challenges on colleges.college_id = challenges.college_id
LEFT JOIN (SELECT challenge_id, 
                    SUM(total_views) AS total_views, 
                    SUM(total_unique_views) AS total_unique_views 
            FROM view_stats 
            GROUP BY challenge_id) AS view_stats
    ON challenges.challenge_id = view_stats.challenge_id
LEFT JOIN (SELECT challenge_id, 
            SUM(total_submissions) AS total_submissions, 
            SUM(total_accepted_submissions) AS total_accepted_submissions 
            FROM Submission_Stats 
            GROUP BY challenge_id) AS submission_stats 
    ON challenges.challenge_id = submission_stats.challenge_id
GROUP BY contests.contest_id, contests.hacker_id, contests.name
HAVING  SUM(submission_stats.total_submissions) != 0 AND
        SUM(submission_stats.total_accepted_submissions)!= 0 AND
        SUM(view_stats.total_views)!= 0 AND
        SUM(view_stats.total_unique_views) != 0
ORDER BY contests.contest_id""", con)

print(df)



   contest_id hacker_id       name  total_subs  total_accepted_subs  \
0       10568      8802      Kelly        1907                  620   
1       11100      8809      Robin        1929                  613   
2       12742      9203      Ralph        1523                  413   
3       12861      9644     Gloria        1596                  536   
4       12865     10108     Victor        2076                  597   
5       13503     10803      David         924                  251   
6       13537     11390      Joyce        1381                  497   
7       13612     12592      Donna        1981                  550   
8       14502     12923   Michelle        1510                  463   
9       14867     13017  Stephanie        2471                  676   
10      15164     13256     Gerald        2570                  820   
11      15804     13421     Walter        1454                  459   
12      15891     13569  Christina        2188                  710   
13    